<a href="https://colab.research.google.com/github/diyamofficial/Internship_June2026/blob/main/RAG_Phase1_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Basic RAG Pipeline using Gemini Embeddings and Groq**

This notebook demonstrates the core concepts of Retrieval-Augmented Generation (RAG):

- Document Chunking
- Text Embeddings
- Similarity Search
- Context Retrieval
- Answer Generation using Groq

**Chunking** breaks a large document into smaller pieces so the system can find relevant information more accurately.
Example:
A 100-page Japanese cuisine book is split into small sections like Sushi, Ramen, Tempura, etc., instead of searching the entire book at once.

**Embeddings** convert text into numbers that capture its meaning, allowing the computer to understand semantic relationships.
Example:
Even though "sushi" and "raw fish dish" are different words, their embeddings will be close because they have similar meanings.

**Cosine similarity** is a metric used to find how closely the meaning of a user's query matches the meaning of stored document chunks by comparing the angle between their embedding vectors.

1 → very similar (same direction)
0 → unrelated (perpendicular)
-1 → completely opposite

Do these two sentences point in the same direction in meaning space?

In [ ]:
# Install Groq library
# To connect our application with Groq's LLM API.
!pip install groq -q

from groq import Groq
from google.colab import userdata

# Read API key from Colab Secrets
# Creates a connection to the Groq API using the stored API key. All LLM requests will be sent through this client.
client = Groq(api_key=userdata.get("GROQ_KEY"))

# Sends a question to the Llama 3.3 70B model hosted on Groq.To verify that LLM is working, before integrating to our pipeline
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "What is RAG in AI?"}
    ]
)

print(response.choices[0].message.content)

In [ ]:
# Sentence Transformer Library-To convert text into embeddings
!pip install sentence-transformers -q

In [ ]:
from sentence_transformers import SentenceTransformer

# Load pretrained embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "I am a Computer Science Engineering student at Mar Baselios College of Engineering and Technology",
    "Diya enjoys travelling,grooving to music",
    "It's beautiful when it rains",
    "Machine learning helps computers learn from data",
    "Varkala has become the new goa of south india"
]

embeddings = model.encode(sentences)

print("Number of embeddings:", len(embeddings))
print("Dimension of one embedding:", len(embeddings[0]))

In [ ]:
print(embeddings[0][:10])
print(embeddings.shape)

In [ ]:
# It converts the query into an embedding and computes cosine similarity with all stored embeddings to find which sentences are most semantically similar.
from sentence_transformers import util

query = "Who studies Computer Science?"

query_embedding = model.encode(query)

scores = util.cos_sim(query_embedding, embeddings)

print(scores)

In [ ]:
import numpy as np

best_match = np.argmax(scores)

print("Query:", query)
print("Best Match:", sentences[best_match])

**A basic RAG workflow by chunking a sample document, retrieving relevant chunks, and passing the retrieved context to Groq for answer generation.**

In [ ]:
# Sample knowledge base document
document = """
Japanese cuisine encompasses the regional and traditional foods of Japan, which have developed through centuries of political, economic, and social changes. The traditional cuisine of Japan (washoku) is based on rice with miso soup and other dishes with an emphasis on seasonal ingredients. Side dishes often consist of fish, pickled vegetables, tamagoyaki, and vegetables cooked in broth. Common seafood is often grilled, but it is also sometimes served raw as sashimi or as sushi. Seafood and vegetables are also deep-fried in a light batter, as tempura. Apart from rice, a staple includes noodles, such as soba and udon. Japan also has many simmered dishes, such as fish products in broth called oden, or beef in sukiyaki and nikujaga.

Historically influenced by Chinese cuisine, Japanese cuisine has also opened up to influence from Western cuisines in the modern era. Dishes inspired by foreign food, in particular Chinese food, like ramen and gyoza, as well as foods like spaghetti, curry and hamburgers, have been adapted to Japanese tastes and ingredients. Traditionally, the Japanese shunned meat as a result of adherence to Buddhism, but with the modernization of Japan in the 1880s, meat-based dishes such as tonkatsu and yakiniku have become common. Since this time, Japanese cuisine, particularly sushi and ramen, has become popular globally.

In 2011, Japan overtook France to become the country with the most 3-starred Michelin restaurants. As of 2018, Tokyo maintained the title of the city with the most 3-starred restaurants in the world. In 2013, Japanese cuisine was added to the UNESCO Intangible Heritage List.
"""

In [ ]:
# It splits a document into paragraph chunks using blank lines and prints each chunk with numbering for easy analysis.
chunks = document.split("\n\n")

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print("-" * 50)

In [ ]:
# Generate Embeddings
chunk_embeddings = model.encode(chunks)

print(chunk_embeddings.shape)

In [ ]:
# My query & Embedding query
query = "I love japanese cuisine.Why?"

query_embedding = model.encode(query)

# Similarity Search

scores = util.cos_sim(query_embedding, chunk_embeddings)

for i, score in enumerate(scores[0]):
    print(f"Chunk {i+1}: {score:.4f}")

In [ ]:
# Displaying Relevant Chunks
scores = scores.numpy()[0]

top_indices = np.argsort(scores)[::-1][:3]

for rank, idx in enumerate(top_indices, start=1):
    print(f"Result {rank}")
    print(chunks[idx])
    print()

In [ ]:
# merges them into a single readable context.
retrieved_chunks = "\n\n".join([chunks[idx] for idx in top_indices])

print(retrieved_chunks)

In [ ]:
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get("GROQ_KEY"))

In [ ]:
# Send Context to Groq and get Final Answer
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": "Answer only using the provided context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{retrieved_chunks}

Question:
{query}

Answer:
"""
        }
    ]
)

print(response.choices[0].message.content)